In [ ]:
import os
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys
sys.path.append('../')

In [ ]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer, param_utils
import parametrization, benchmark
import numpy as np

In [ ]:
import continuation_parametrization, flip_avoiding_step_length

In [ ]:
from Benchmark import helper_funcs

# Read Mesh and initialization

In [ ]:
mesh_path = '../../../Models/PPdata/data1/fig12_a.obj'

In [ ]:
m = helper_funcs.read_mesh(mesh_path)

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
uv_init = helper_funcs.tutteInitialization(m, bdry_uv)

In [ ]:
uv.setVars(uv_init.ravel())

# Prob Set up

In [ ]:
param = continuation_parametrization.symmetric_dirichlet_param(m, uv)
objectives = [param]

In [ ]:
# Construct parametrization energy and problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)
opt = prob.optimizer()

In [ ]:
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.8

In [ ]:
# prob.hessianShift = 1e-9
# prob.useRelativeHessianShift = True

param.elementHessianShift = 1e-6

In [ ]:
prob.objective()

In [ ]:
DEGREE = 0

In [ ]:
# opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
# opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
# opt.options.hessianProjectionController.startWithProjectionActive = False

opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
# benchmark.reset()
# param.setInterpolatedReference(0, uv_init.ravel())
# opt.options.hessianProjectionController.reset()
# prob.invalidateCachedHessian()
# opt.update_factorizations()
# print(prob.hessianWasProjected)
# benchmark.report()

## read l from file

In [ ]:
l_arr = np.loadtxt('match_CM_true_area/fig12_a_pp_t.txt')

In [ ]:
stats_trace = []

In [ ]:
benchmark.reset()

# param.setInterpolatedReference(0, uv_init.ravel())
prob.setVars(uv_init.ravel())

for it, l in enumerate(l_arr):
    
    print('l: ', l)
    # param.setInterpolatedReference(l, prob.getVars())
    param.rebaseInterpolatedReference(l, prob.getVars())
    
    prob.invalidateCachedHessian()
    # opt.options.hessianProjectionController.reset()
    opt.update_factorizations()
    
    x0 = prob.getVars()
    opt.options.niter = 1
    opt.options.gradTol = 1e-6
    opt.optimize()
    
    # eval prob at 1.0
    param.setInterpolatedReference(1.0, uv_init.ravel())
    stats_trace.append({
        'iter': it,
        'l': float(l),
        'energy': prob.energy(),
        'grad_norm': np.linalg.norm(prob.gradient()),
    })
    # print(f'energy: {prob.energy()}; gradient_norm: {np.linalg.norm(prob.gradient())}')

benchmark.report()


# Debug

In [ ]:
for st in stats_trace:
    print(f"{st['iter']}    {st['l']}    {st['energy']}    {st['grad_norm']:.5e}")

In [ ]:
# TODO:
# Try attenuating the Hessian projection amount (interpolate down to no projection)